# 📈 Forecasting Models — Time Series Machine Learning

> **Folder:** `09_Time_Series_Machine_Learning`  
> **Notebook:** `forecasting_models.ipynb`  
> **Author:** Hamna Munir

---

## 🎯 Objectives

By the end of this notebook, you will:

- Build **baseline forecasting models** (Naive, Seasonal Naive, Moving Average)
- Implement **ARIMA / SARIMA** for classical statistical forecasting
- Use **Prophet** for trend + seasonality + holiday decomposition
- Apply **ML models** (GBM, RF, XGBoost) to tabular time series features
- Compare all models on a unified **leaderboard with proper metrics**
- Generate **multi-step forecasts** with recursive and direct strategies
- Build **prediction intervals** for uncertainty quantification

---

## 📚 Techniques Covered

| # | Technique | Key Insight |
|---|-----------|-------------|
| 1 | Dataset Setup | Synthetic + real-structure daily series |
| 2 | Baseline Models | Naive, Seasonal Naive, Moving Average |
| 3 | ARIMA / SARIMA | Classical statistical model |
| 4 | Exponential Smoothing | Holt-Winters method |
| 5 | Prophet | Facebook's decomposition forecaster |
| 6 | ML Forecasting Setup | Feature matrix + TimeSeriesSplit |
| 7 | GBM / RF Forecasting | Tree-based ML for tabular TS |
| 8 | Multi-Step Forecasting | Recursive vs Direct strategies |
| 9 | Forecast Evaluation | MAE, RMSE, MAPE, SMAPE |
| 10 | Prediction Intervals | Uncertainty quantification |
| 11 | Model Leaderboard | Full comparison |
| 12 | Summary & Golden Rules | Key takeaways |


---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    GradientBoostingRegressor, RandomForestRegressor
)
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.stattools import adfuller
    STATSMODELS = True
except ImportError:
    STATSMODELS = False
    print("statsmodels not installed — pip install statsmodels")

try:
    from prophet import Prophet
    PROPHET = True
except ImportError:
    try:
        from fbprophet import Prophet
        PROPHET = True
    except ImportError:
        PROPHET = False
        print("Prophet not installed — pip install prophet")

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

COLORS = {
    'primary'  : '#2E86AB',
    'secondary': '#E84855',
    'accent'   : '#3BB273',
    'warning'  : '#F18F01',
    'purple'   : '#7B2D8B',
    'palette'  : ['#2E86AB','#E84855','#3BB273','#F18F01','#7B2D8B','#F4D35E'],
}
print('✅ Libraries loaded!')
print(f'  statsmodels: {STATSMODELS} | Prophet: {PROPHET} | XGBoost: {XGB_AVAILABLE}')

---
## 1️⃣ Dataset Setup — Synthetic Daily Sales Series

> Realistic synthetic series: trend + weekly + yearly seasonality + noise.  
> **Temporal split:** last 90 days = test set (held out for final evaluation).


In [ ]:
np.random.seed(42)

# ── Build 3-year daily series ─────────────────────────────────────────────
n_days = 365 * 3
dates  = pd.date_range(start='2021-01-01', periods=n_days, freq='D')
t      = np.arange(n_days)

trend    = 200 + 0.08 * t
weekly   = 20  * np.sin(2 * np.pi * t / 7)
yearly   = 40  * np.sin(2 * np.pi * t / 365.25 - np.pi/2)
noise    = np.random.normal(0, 12, n_days)
value    = np.maximum(trend + weekly + yearly + noise, 10)  # no negatives

df = pd.DataFrame({'y': value}, index=dates)
df.index.name = 'ds'
df.index.freq = 'D'

# ── Temporal split ────────────────────────────────────────────────────────
test_days  = 90
train_df   = df.iloc[:-test_days]
test_df    = df.iloc[-test_days:]

print(f'Full series   : {df.shape[0]} days | {df.index[0].date()} to {df.index[-1].date()}')
print(f'Train         : {train_df.shape[0]} days')
print(f'Test (holdout): {test_df.shape[0]} days')
print(f'\nTarget stats:')
print(df['y'].describe().round(2).to_string())

# Plot train/test split
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(train_df.index, train_df['y'], color=COLORS['primary'],
        linewidth=1.0, label=f'Train ({len(train_df)} days)')
ax.plot(test_df.index, test_df['y'], color=COLORS['secondary'],
        linewidth=1.5, label=f'Test ({len(test_df)} days)')
ax.axvline(test_df.index[0], color='black', linestyle='--',
           linewidth=2, label='Train/Test split')
ax.set_ylabel('Value', fontsize=11)
ax.set_title('Synthetic Daily Series — Train / Test Split',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout(); plt.show()

---
## 2️⃣ Baseline Forecasting Models

> Always benchmark against simple baselines before using complex models.  
> If your ML model can't beat a Naive forecast — something is wrong.
>
> | Baseline | Formula | When to Use |
> |---------|---------|-------------|
> | **Naive** | ŷₜ = yₜ₋₁ | No trend, no seasonality |
> | **Seasonal Naive** | ŷₜ = yₜ₋ₛ (s=period) | Strong seasonality, no trend |
> | **Moving Average** | ŷₜ = mean(yₜ₋w,...,yₜ₋₁) | Smoothed level |
> | **Drift** | ŷₜ = yₜ₋₁ + avg Δy | Clear trend |


In [ ]:
def evaluate_forecast(y_true, y_pred, model_name):
    """Compute MAE, RMSE, MAPE, SMAPE for a forecast."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mae    = mean_absolute_error(y_true, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mape   = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    smape  = np.mean(2 * np.abs(y_true - y_pred) /
                     (np.abs(y_true) + np.abs(y_pred))) * 100
    r2     = r2_score(y_true, y_pred)
    return {'Model': model_name, 'MAE': round(mae,3), 'RMSE': round(rmse,3),
            'MAPE%': round(mape,3), 'SMAPE%': round(smape,3), 'R2': round(r2,4)}

leaderboard = []
y_true = test_df['y'].values

# ── Naive forecast ────────────────────────────────────────────────────────
naive_pred = np.full(test_days, train_df['y'].iloc[-1])
leaderboard.append(evaluate_forecast(y_true, naive_pred, 'Naive'))

# ── Seasonal Naive (period=7) ─────────────────────────────────────────────
sn_pred = []
for i in range(test_days):
    idx = -(7 - (i % 7))
    sn_pred.append(train_df['y'].iloc[idx])
sn_pred = np.array(sn_pred)
leaderboard.append(evaluate_forecast(y_true, sn_pred, 'Seasonal Naive (s=7)'))

# ── Moving Average (28-day) ───────────────────────────────────────────────
ma_val  = train_df['y'].iloc[-28:].mean()
ma_pred = np.full(test_days, ma_val)
leaderboard.append(evaluate_forecast(y_true, ma_pred, 'Moving Average (28d)'))

# ── Drift ─────────────────────────────────────────────────────────────────
avg_drift  = (train_df['y'].iloc[-1] - train_df['y'].iloc[0]) / len(train_df)
drift_pred = np.array([train_df['y'].iloc[-1] + avg_drift * (i+1)
                       for i in range(test_days)])
leaderboard.append(evaluate_forecast(y_true, drift_pred, 'Drift'))

print('Baseline Model Performance:')
lb_df = pd.DataFrame(leaderboard)
print(lb_df.to_string(index=False))

# Plot baselines
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(train_df.index[-60:], train_df['y'].iloc[-60:],
        color=COLORS['primary'], linewidth=1.5, label='Train (last 60d)')
ax.plot(test_df.index, y_true, color='black',
        linewidth=2.0, label='Actual', zorder=5)
for pred, name, color in [
    (naive_pred, 'Naive', COLORS['secondary']),
    (sn_pred,   'Seasonal Naive', COLORS['accent']),
    (ma_pred,   'Moving Avg (28d)', COLORS['warning']),
    (drift_pred,'Drift', COLORS['purple']),
]:
    ax.plot(test_df.index, pred, linewidth=1.8,
            color=color, linestyle='--', label=name)
ax.axvline(test_df.index[0], color='black', linestyle=':', linewidth=1.5)
ax.set_title('Baseline Forecasts vs Actual', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.set_ylabel('Value')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout(); plt.show()

---
## 3️⃣ ARIMA / SARIMA — Classical Statistical Forecasting

> **ARIMA(p,d,q):**
> - p = AR order (autoregressive lags)
> - d = differencing order (for stationarity)
> - q = MA order (moving average lags)
>
> **SARIMA(p,d,q)(P,D,Q,s):** adds seasonal AR, diff, MA terms with period s.
>
> Rule of thumb for this series: `SARIMA(1,1,1)(1,1,1,7)` — weekly seasonality.


In [ ]:
if STATSMODELS:
    print('Fitting SARIMA(1,1,1)(1,1,1,7)...')
    sarima_model = SARIMAX(
        train_df['y'],
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    sarima_fit = sarima_model.fit(disp=False)

    # Forecast
    sarima_forecast = sarima_fit.forecast(steps=test_days)
    sarima_pred     = sarima_forecast.values
    leaderboard.append(evaluate_forecast(y_true, sarima_pred, 'SARIMA(1,1,1)(1,1,1,7)'))

    # Prediction intervals
    pred_summary = sarima_fit.get_forecast(steps=test_days).summary_frame()
    ci_lower     = pred_summary['mean_ci_lower'].values
    ci_upper     = pred_summary['mean_ci_upper'].values

    print(f'SARIMA AIC : {sarima_fit.aic:.2f}')
    print(f'SARIMA BIC : {sarima_fit.bic:.2f}')
    print(evaluate_forecast(y_true, sarima_pred, 'SARIMA(1,1,1)(1,1,1,7)'))

    # Plot
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.plot(train_df.index[-90:], train_df['y'].iloc[-90:],
            color=COLORS['primary'], linewidth=1.5, label='Train (last 90d)')
    ax.plot(test_df.index, y_true, color='black',
            linewidth=2, label='Actual')
    ax.plot(test_df.index, sarima_pred, color=COLORS['secondary'],
            linewidth=2, linestyle='--', label='SARIMA forecast')
    ax.fill_between(test_df.index, ci_lower, ci_upper,
                    alpha=0.18, color=COLORS['secondary'],
                    label='95% Prediction Interval')
    ax.axvline(test_df.index[0], color='black', linestyle=':', linewidth=1.5)
    ax.set_title('SARIMA(1,1,1)(1,1,1,7) — Forecast with Prediction Intervals',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10); ax.set_ylabel('Value')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    plt.tight_layout(); plt.show()
else:
    print('statsmodels not available — skipping SARIMA')
    print('pip install statsmodels')

---
## 4️⃣ Exponential Smoothing — Holt-Winters Method

> Holt-Winters exponential smoothing models:
> - **Level** (α): current baseline
> - **Trend** (β): direction of change
> - **Seasonality** (γ): periodic patterns
>
> Additive seasonality: constant seasonal amplitude  
> Multiplicative seasonality: seasonal amplitude grows with level


In [ ]:
if STATSMODELS:
    # Holt-Winters — additive trend, additive seasonality
    hw_model = ExponentialSmoothing(
        train_df['y'],
        trend='add',
        seasonal='add',
        seasonal_periods=7,
        damped_trend=True,
    )
    hw_fit   = hw_model.fit(optimized=True)
    hw_pred  = hw_fit.forecast(test_days).values
    leaderboard.append(evaluate_forecast(y_true, hw_pred, 'Holt-Winters (add+add)'))

    print('Holt-Winters Parameters:')
    print(f'  alpha (level)      : {hw_fit.params["smoothing_level"]:.4f}')
    print(f'  beta  (trend)      : {hw_fit.params["smoothing_trend"]:.4f}')
    print(f'  gamma (seasonal)   : {hw_fit.params["smoothing_seasonal"]:.4f}')
    print(f'  phi   (damping)    : {hw_fit.params.get("damping_trend", "N/A")}')
    print()
    print(evaluate_forecast(y_true, hw_pred, 'Holt-Winters'))

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(train_df.index[-90:], train_df['y'].iloc[-90:],
            color=COLORS['primary'], linewidth=1.5, label='Train (last 90d)')
    ax.plot(test_df.index, y_true, color='black', linewidth=2, label='Actual')
    ax.plot(test_df.index, hw_pred, color=COLORS['accent'],
            linewidth=2, linestyle='--', label='Holt-Winters')
    ax.axvline(test_df.index[0], color='black', linestyle=':', linewidth=1.5)
    ax.set_title('Holt-Winters Exponential Smoothing — Forecast',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10); ax.set_ylabel('Value')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    plt.tight_layout(); plt.show()
else:
    print('statsmodels not available — skipping Holt-Winters')

---
## 5️⃣ Prophet — Trend + Seasonality + Holidays

> **Prophet** (Meta/Facebook) is designed for business time series with:
> - Multiple seasonality components (weekly, yearly)
> - Holiday effects (custom events)
> - Automatic changepoint detection in trend
> - Uncertainty intervals via Monte Carlo
>
> Input format: DataFrame with columns `ds` (datetime) and `y` (value).


In [ ]:
if PROPHET:
    # Format for Prophet
    prophet_train = train_df.reset_index().rename(columns={'ds':'ds','y':'y'})
    prophet_train.columns = ['ds', 'y']

    prophet_model = Prophet(
        seasonality_mode='additive',
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,   # flexibility of trend (higher = more flexible)
        seasonality_prior_scale=10.0,   # strength of seasonality
        interval_width=0.95,
    )
    prophet_model.fit(prophet_train)

    # Future DataFrame
    future = prophet_model.make_future_dataframe(periods=test_days, freq='D')
    forecast = prophet_model.predict(future)

    prophet_pred = forecast.iloc[-test_days:]['yhat'].values
    prophet_lower = forecast.iloc[-test_days:]['yhat_lower'].values
    prophet_upper = forecast.iloc[-test_days:]['yhat_upper'].values

    leaderboard.append(evaluate_forecast(y_true, prophet_pred, 'Prophet'))
    print(evaluate_forecast(y_true, prophet_pred, 'Prophet'))

    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))

    axes[0].plot(train_df.index[-90:], train_df['y'].iloc[-90:],
                 color=COLORS['primary'], linewidth=1.5, label='Train (last 90d)')
    axes[0].plot(test_df.index, y_true, color='black',
                 linewidth=2, label='Actual')
    axes[0].plot(test_df.index, prophet_pred, color=COLORS['purple'],
                 linewidth=2, linestyle='--', label='Prophet forecast')
    axes[0].fill_between(test_df.index, prophet_lower, prophet_upper,
                         alpha=0.15, color=COLORS['purple'],
                         label='95% Prediction Interval')
    axes[0].axvline(test_df.index[0], color='black', linestyle=':', linewidth=1.5)
    axes[0].set_title('Prophet — Forecast with Uncertainty Intervals',
                      fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=9); axes[0].set_ylabel('Value')

    # Component plot (trend, weekly, yearly)
    trend_vals   = forecast['trend'].iloc[-test_days-365:-test_days+30]
    weekly_vals  = forecast['weekly'].iloc[-test_days-30:-test_days+30]
    axes[1].plot(forecast['ds'].iloc[-test_days-365:-test_days+30],
                 trend_vals, color=COLORS['primary'], linewidth=2, label='Trend')
    axes[1].set_title('Prophet — Trend Component', fontsize=12, fontweight='bold')
    axes[1].legend(fontsize=9); axes[1].set_ylabel('Trend')

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    plt.suptitle('Prophet Forecasting', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('Prophet not installed — pip install prophet')
    print('Skipping Prophet cell.')

---
## 6️⃣ ML Forecasting Setup — Feature Matrix

> ML models treat forecasting as a **supervised regression** problem.  
> We build a feature matrix from lag and calendar features,  
> then train any sklearn regressor on it.


In [ ]:
def build_features(df, horizon=1, lags=None, windows=None):
    """
    Build ML feature matrix for time series forecasting.
    All features respect the forecast horizon (no lookahead).

    Args:
        df      : DataFrame with 'y' column and DatetimeIndex
        horizon : Forecast horizon (h steps ahead)
        lags    : List of lag values to include
        windows : List of rolling window sizes

    Returns:
        X (features), y (target), feature names
    """
    if lags is None:
        lags = [1, 2, 3, 7, 14, 21, 28]
    if windows is None:
        windows = [7, 14, 28, 90]

    feat = df.copy()

    # Lag features (must be >= horizon)
    for lag in lags:
        if lag >= horizon:
            feat[f'lag_{lag}'] = feat['y'].shift(lag)

    # Rolling features (shift by horizon first)
    base = feat['y'].shift(horizon)
    for w in windows:
        feat[f'roll_mean_{w}'] = base.rolling(w).mean()
        feat[f'roll_std_{w}']  = base.rolling(w).std()
        feat[f'roll_min_{w}']  = base.rolling(w).min()
        feat[f'roll_max_{w}']  = base.rolling(w).max()

    # Expanding features
    feat['exp_mean'] = base.expanding().mean()
    feat['exp_std']  = base.expanding().std()

    # Calendar features (cyclical)
    feat['dow_sin']   = np.sin(2 * np.pi * feat.index.dayofweek / 7)
    feat['dow_cos']   = np.cos(2 * np.pi * feat.index.dayofweek / 7)
    feat['month_sin'] = np.sin(2 * np.pi * feat.index.month / 12)
    feat['month_cos'] = np.cos(2 * np.pi * feat.index.month / 12)
    feat['doy_sin']   = np.sin(2 * np.pi * feat.index.dayofyear / 365.25)
    feat['doy_cos']   = np.cos(2 * np.pi * feat.index.dayofyear / 365.25)
    feat['is_weekend']= (feat.index.dayofweek >= 5).astype(int)

    # Target = y shifted back by horizon
    feat['target'] = feat['y'].shift(-horizon)

    feature_cols = [c for c in feat.columns if c not in ['y', 'target']]
    feat = feat.dropna(subset=feature_cols + ['target'])

    return feat[feature_cols], feat['target'], feature_cols

# Build features for horizon=1
h = 1
X_all, y_all, feat_names = build_features(df, horizon=h)

# Temporal split aligned with our train/test dates
split_idx  = len(X_all) - test_days
X_train_ml = X_all.iloc[:split_idx]
y_train_ml = y_all.iloc[:split_idx]
X_test_ml  = X_all.iloc[split_idx:]
y_test_ml  = y_all.iloc[split_idx:]

print(f'Feature matrix: {X_all.shape}')
print(f'Train: {X_train_ml.shape} | Test: {X_test_ml.shape}')
print(f'Features ({len(feat_names)}): {feat_names[:8]} ...')

---
## 7️⃣ ML Forecasting — GBM / RF / XGBoost

> Tree-based ML models work well for time series forecasting because:
> - They handle non-linear relationships automatically
> - No stationarity requirement
> - Feature importance for interpretability
> - Easy to include external regressors (weather, events, etc.)


In [ ]:
scaler_ml = StandardScaler()
X_tr_sc   = scaler_ml.fit_transform(X_train_ml)
X_te_sc   = scaler_ml.transform(X_test_ml)

ml_models = {
    'Ridge'            : Ridge(alpha=10.0),
    'RandomForest'     : RandomForestRegressor(
        n_estimators=200, max_depth=8, min_samples_leaf=3,
        random_state=42, n_jobs=-1),
    'GradientBoosting' : GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, min_samples_leaf=3, random_state=42),
}

if XGB_AVAILABLE:
    ml_models['XGBoost'] = xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        tree_method='hist', random_state=42, verbosity=0)

ml_predictions = {}
print('ML Model Forecasting Results (h=1):')
for name, model in ml_models.items():
    model.fit(X_tr_sc, y_train_ml)
    y_pred = model.predict(X_te_sc)
    ml_predictions[name] = y_pred
    metrics = evaluate_forecast(y_test_ml.values, y_pred, name)
    leaderboard.append(metrics)
    print(f'  {name:20s}: MAE={metrics["MAE"]:.3f} | '
          f'RMSE={metrics["RMSE"]:.3f} | MAPE={metrics["MAPE%"]:.2f}%')

# Plot predictions
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(train_df.index[-60:], train_df['y'].iloc[-60:],
        color=COLORS['primary'], linewidth=1.5, label='Train (last 60d)')
ax.plot(test_df.index[:len(y_test_ml)], y_test_ml.values,
        color='black', linewidth=2.0, label='Actual', zorder=5)

for (name, pred), color in zip(ml_predictions.items(), COLORS['palette'][1:]):
    ax.plot(test_df.index[:len(pred)], pred,
            linewidth=1.8, color=color, linestyle='--',
            alpha=0.85, label=name)

ax.axvline(test_df.index[0], color='black', linestyle=':', linewidth=1.5)
ax.set_title('ML Model Forecasts vs Actual (h=1)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.set_ylabel('Value')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout(); plt.show()

# Feature importance (GBM)
gbm_imp = pd.Series(
    ml_models['GradientBoosting'].feature_importances_,
    index=feat_names
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 5))
ax.barh(gbm_imp.index[:15], gbm_imp.values[:15],
        color=COLORS['primary'], alpha=0.85, edgecolor='white')
ax.set_xlabel('Feature Importance (MDI)', fontsize=11)
ax.set_title('GBM — Top 15 Feature Importances for Forecasting',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8️⃣ Multi-Step Forecasting — Recursive vs Direct Strategies

> **Recursive (one-step-ahead chained):**
> - Train one model for h=1
> - Forecast one step → feed as input → forecast next → repeat
> - Error accumulates over horizon
>
> **Direct (separate model per horizon):**
> - Train one model per horizon h=1,2,...,H
> - Each model is specialized for its horizon
> - No error accumulation — but needs H separate models


In [ ]:
H = 14  # forecast horizon (14 steps ahead)

# ── Recursive strategy ────────────────────────────────────────────────────
def recursive_forecast(train_df, model, scaler, feat_names, H, horizon=1):
    """Recursive (chained) multi-step forecast."""
    history = train_df.copy()
    preds   = []

    for step in range(H):
        X_now, _, _ = build_features(history, horizon=horizon)
        if len(X_now) == 0:
            break
        X_now_sc = scaler.transform(X_now.iloc[[-1]])
        y_hat    = model.predict(X_now_sc)[0]
        preds.append(y_hat)

        # Append prediction to history
        next_date   = history.index[-1] + pd.Timedelta(days=1)
        new_row     = pd.DataFrame({'y': [y_hat]}, index=[next_date])
        history     = pd.concat([history, new_row])

    return np.array(preds)

# ── Direct strategy ───────────────────────────────────────────────────────
def direct_forecast(train_df, H):
    """Direct multi-step: separate GBM per horizon."""
    preds   = []
    full_df = pd.concat([train_df, test_df])  # full data for feature building

    for h in range(1, H+1):
        X_h, y_h, _ = build_features(train_df, horizon=h)
        sc_h        = StandardScaler()
        X_h_sc      = sc_h.fit_transform(X_h)

        model_h = GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05,
            max_depth=4, subsample=0.8, random_state=42
        )
        model_h.fit(X_h_sc, y_h)

        # Predict for the test period at this horizon
        X_pred, _, _ = build_features(
            pd.concat([train_df, test_df.iloc[:h]]), horizon=h
        )
        X_pred_sc = sc_h.transform(X_pred.iloc[[-1]])
        preds.append(model_h.predict(X_pred_sc)[0])

    return np.array(preds)

print('Generating multi-step forecasts...')
recursive_pred = recursive_forecast(
    train_df, ml_models['GradientBoosting'], scaler_ml, feat_names, H)
direct_pred    = direct_forecast(train_df, H)
actual_H       = test_df['y'].values[:H]

rec_metrics = evaluate_forecast(actual_H, recursive_pred, f'Recursive (H={H})')
dir_metrics = evaluate_forecast(actual_H, direct_pred, f'Direct (H={H})')
print(f'Recursive: {rec_metrics}')
print(f'Direct   : {dir_metrics}')

fig, ax = plt.subplots(figsize=(14, 5))
horizon_range = range(1, H+1)
ax.plot(horizon_range, actual_H, color='black',
        linewidth=2.5, marker='o', markersize=5, label='Actual', zorder=5)
ax.plot(horizon_range, recursive_pred, color=COLORS['primary'],
        linewidth=2, marker='s', markersize=5, linestyle='--',
        label=f'Recursive (RMSE={rec_metrics["RMSE"]:.2f})')
ax.plot(horizon_range, direct_pred, color=COLORS['secondary'],
        linewidth=2, marker='^', markersize=5, linestyle='--',
        label=f'Direct (RMSE={dir_metrics["RMSE"]:.2f})')
ax.set_xlabel('Forecast Horizon (days ahead)', fontsize=11)
ax.set_ylabel('Value', fontsize=11)
ax.set_title(f'Multi-Step Forecasting — Recursive vs Direct (H={H})',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.set_xticks(horizon_range)
plt.tight_layout(); plt.show()

---
## 9️⃣ Forecast Evaluation Metrics

> | Metric | Formula | Notes |
> |--------|---------|-------|
> | **MAE** | mean |y−ŷ| | Interpretable, same unit as y |
> | **RMSE** | √(mean(y−ŷ)²) | Penalizes large errors more |
> | **MAPE** | mean |y−ŷ|/|y| × 100 | Scale-free %, undefined at y=0 |
> | **SMAPE** | mean 2|y−ŷ|/(|y|+|ŷ|) × 100 | Symmetric MAPE, bounded [0,200] |
> | **R²** | 1 − SS_res/SS_tot | Fraction of variance explained |


In [ ]:
# Plot error metrics across all models
all_metrics = pd.DataFrame([m for m in leaderboard
                              if 'Recursive' not in m['Model']
                              and 'Direct' not in m['Model']
                             ]).sort_values('RMSE')

print('All Models — Forecast Metrics (sorted by RMSE):')
print(all_metrics.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(17, 10))

for ax, metric, color in zip(axes.flatten(),
    ['MAE','RMSE','MAPE%','SMAPE%'],
    COLORS['palette'][:4]):
    sorted_m = all_metrics.sort_values(metric)
    bar_colors = [COLORS['accent'] if m in ['GradientBoosting','RandomForest',
                                              'XGBoost','Ridge']
                  else COLORS['primary'] for m in sorted_m['Model']]
    bars = ax.barh(sorted_m['Model'], sorted_m[metric],
                   color=bar_colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, sorted_m[metric]):
        ax.text(bar.get_width()+0.02, bar.get_y()+bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=9)
    ax.set_xlabel(metric, fontsize=11)
    ax.set_title(f'Models Ranked by {metric}', fontsize=11, fontweight='bold')

plt.suptitle('Forecast Evaluation — All Models All Metrics',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Error distribution for best ML model
best_ml = 'GradientBoosting'
gbm_errors = y_test_ml.values - ml_predictions[best_ml]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(gbm_errors, bins=25, color=COLORS['primary'],
             alpha=0.80, edgecolor='white')
axes[0].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[0].axvline(gbm_errors.mean(), color=COLORS['secondary'],
                linestyle='--', linewidth=2,
                label=f'Mean error={gbm_errors.mean():.2f}')
axes[0].set_xlabel('Forecast Error (actual - predicted)', fontsize=11)
axes[0].set_title(f'{best_ml} — Error Distribution', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].scatter(ml_predictions[best_ml], gbm_errors,
                alpha=0.4, s=20, color=COLORS['primary'])
axes[1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Value', fontsize=11)
axes[1].set_ylabel('Residual', fontsize=11)
axes[1].set_title(f'{best_ml} — Residuals vs Predicted', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🔟 Prediction Intervals — Uncertainty Quantification

> A forecast without uncertainty bounds is incomplete.  
> Prediction intervals capture the range of likely future values.
>
> Methods:
> - **Quantile regression:** train separate models for low and high quantiles
> - **Conformal prediction:** coverage-guaranteed intervals from residuals
> - **Bootstrap:** resample residuals to estimate distribution


In [ ]:
# ── Conformal Prediction Intervals ───────────────────────────────────────
# Simple residual-based approach using calibration set
n_calib     = 60
X_tr_calib  = X_tr_sc[:-n_calib]
y_tr_calib  = y_train_ml.values[:-n_calib]
X_calib     = X_tr_sc[-n_calib:]
y_calib     = y_train_ml.values[-n_calib:]

gbm_calib   = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, random_state=42
)
gbm_calib.fit(X_tr_calib, y_tr_calib)

# Calibration residuals
calib_preds  = gbm_calib.predict(X_calib)
calib_errors = np.abs(y_calib - calib_preds)

# Conformal quantile (95% coverage)
alpha      = 0.05
q_hat      = np.quantile(calib_errors, 1 - alpha)
test_preds = gbm_calib.predict(X_te_sc)
ci_lower   = test_preds - q_hat
ci_upper   = test_preds + q_hat

# Coverage check
coverage = np.mean((y_test_ml.values >= ci_lower) &
                   (y_test_ml.values <= ci_upper)) * 100
avg_width = np.mean(ci_upper - ci_lower)

print(f'Conformal Prediction Intervals (target: 95%)')
print(f'  Quantile (q_hat) : {q_hat:.3f}')
print(f'  Empirical coverage: {coverage:.1f}% (target: 95%)')
print(f'  Mean interval width: {avg_width:.3f}')

# ── Quantile GBM ─────────────────────────────────────────────────────────
gbm_q10 = GradientBoostingRegressor(loss='quantile', alpha=0.10,
                                      n_estimators=200, learning_rate=0.05,
                                      max_depth=4, random_state=42)
gbm_q50 = GradientBoostingRegressor(loss='quantile', alpha=0.50,
                                      n_estimators=200, learning_rate=0.05,
                                      max_depth=4, random_state=42)
gbm_q90 = GradientBoostingRegressor(loss='quantile', alpha=0.90,
                                      n_estimators=200, learning_rate=0.05,
                                      max_depth=4, random_state=42)
for q_model in [gbm_q10, gbm_q50, gbm_q90]:
    q_model.fit(X_tr_sc, y_train_ml)

q10_pred = gbm_q10.predict(X_te_sc)
q50_pred = gbm_q50.predict(X_te_sc)
q90_pred = gbm_q90.predict(X_te_sc)
q_cov = np.mean((y_test_ml.values >= q10_pred) &
                (y_test_ml.values <= q90_pred)) * 100

# Plot prediction intervals
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Conformal intervals
axes[0].plot(test_df.index[:len(y_test_ml)], y_test_ml.values,
             color='black', linewidth=2, label='Actual')
axes[0].plot(test_df.index[:len(test_preds)], test_preds,
             color=COLORS['primary'], linewidth=2, linestyle='--',
             label='Point forecast (GBM)')
axes[0].fill_between(test_df.index[:len(test_preds)], ci_lower, ci_upper,
                     alpha=0.20, color=COLORS['primary'],
                     label=f'95% Conformal PI (cov={coverage:.1f}%)')
axes[0].set_title('Conformal Prediction Intervals (coverage-guaranteed)',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9); axes[0].set_ylabel('Value')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

# Quantile GBM intervals
axes[1].plot(test_df.index[:len(y_test_ml)], y_test_ml.values,
             color='black', linewidth=2, label='Actual')
axes[1].plot(test_df.index[:len(q50_pred)], q50_pred,
             color=COLORS['secondary'], linewidth=2, linestyle='--',
             label='Median forecast (Q50)')
axes[1].fill_between(test_df.index[:len(q10_pred)], q10_pred, q90_pred,
                     alpha=0.22, color=COLORS['secondary'],
                     label=f'80% Quantile PI (cov={q_cov:.1f}%)')
axes[1].set_title('Quantile Gradient Boosting (Q10-Q50-Q90)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].set_ylabel('Value')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.suptitle('Prediction Intervals — Uncertainty Quantification',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 1️⃣1️⃣ Final Model Leaderboard


In [ ]:
final_lb = pd.DataFrame([m for m in leaderboard
                          if 'Recursive' not in m['Model']
                          and 'Direct' not in m['Model']
                         ]).sort_values('RMSE').reset_index(drop=True)

print('FINAL FORECASTING LEADERBOARD (sorted by RMSE):')
print('=' * 75)
print(final_lb.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 6))
type_colors = {
    'Naive'                     : 'gray',
    'Seasonal Naive (s=7)'      : 'gray',
    'Moving Average (28d)'      : 'gray',
    'Drift'                     : 'gray',
    'SARIMA(1,1,1)(1,1,1,7)'   : COLORS['warning'],
    'Holt-Winters (add+add)'    : COLORS['warning'],
    'Prophet'                   : COLORS['purple'],
    'Ridge'                     : COLORS['primary'],
    'RandomForest'              : COLORS['accent'],
    'GradientBoosting'          : COLORS['secondary'],
    'XGBoost'                   : COLORS['palette'][5],
}
bar_colors = [type_colors.get(m, COLORS['primary']) for m in final_lb['Model']]
bars = ax.barh(final_lb['Model'], final_lb['RMSE'],
               color=bar_colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, final_lb['RMSE']):
    ax.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=10)
ax.set_xlabel('RMSE (lower is better)', fontsize=11)
ax.set_title('Forecasting Model Leaderboard', fontsize=13, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='gray',             label='Baseline'),
    Patch(color=COLORS['warning'],  label='Statistical (ARIMA/HW)'),
    Patch(color=COLORS['purple'],   label='Prophet'),
    Patch(color=COLORS['accent'],   label='ML Models'),
], fontsize=9, loc='lower right')
plt.tight_layout(); plt.show()

---
## ✅ 12. Summary & Golden Rules

| Model Type | Best For | Main Strength | Weakness |
|-----------|---------|--------------|---------|
| **Naive / Seasonal Naive** | Benchmark | Zero parameters | Ignores patterns |
| **Holt-Winters** | Trend + seasonality | Simple, fast | Fixed seasonality |
| **ARIMA / SARIMA** | Stationary series | Statistical rigor | Manual order selection |
| **Prophet** | Business series + holidays | Interpretable, robust | Black-box trend |
| **GBM / XGBoost** | Complex non-linear patterns | High accuracy, flexible | Needs feature engineering |
| **RandomForest** | High-dimensional features | Variance reduction | Doesn't extrapolate trend |

### 🔑 Golden Rules

1. **Always start with baselines** — beat Naive and Seasonal Naive first
2. **Never shuffle time series** — temporal split only
3. **Use TimeSeriesSplit for CV** — no standard K-Fold
4. **Lag features respect the forecast horizon** — lag_k only if k ≥ h
5. **Evaluate with SMAPE and RMSE** — MAPE is unstable near zero
6. **Always include prediction intervals** — point forecasts alone are incomplete
7. **Quantile regression > conformal** when quantiles are the target
8. **Conformal prediction** gives guaranteed coverage — use when unsure
9. **ML models need more data** than ARIMA — prefer ARIMA for short series
10. **Recursive multi-step** is simpler; **direct** is more accurate for long horizons

---

## 🔗 Next Steps

- ➡️ `09_Time_Series_Machine_Learning/preprocessing.md` — Feature preprocessing theory
- ➡️ `09_Time_Series_Machine_Learning/feature_engineering.ipynb` — Lag, rolling, calendar features
- ➡️ `07_Hyperparameter_Tuning/randomsearchcv.ipynb` — Tune GBM for forecasting
- ➡️ `08_Ensemble_Learning/stacking.ipynb` — Stack multiple forecasters
